In [1]:
import pandas as pd
import os
import subprocess
import tempfile
from Bio import SeqIO
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Paths
motif_file = "/home1/smaruj/ledidi_akita/homer_ctcf_elimination/homerResults_meme/motif10RV.meme"
df_path = "/scratch1/smaruj/CTCF_elimination/gamma_300.0/successful_optimizations.tsv"
edited_fasta_dir = "/scratch1/smaruj/CTCF_elimination/gamma_300.0/successful_opts_fasta"
original_fasta_dir = "/scratch1/smaruj/CTCF_elimination/gamma_300.0/OG_seqs_fasta"

In [3]:
# Load successful sequence list
df = pd.read_csv(df_path, sep="\t")

In [4]:
# FIMO parameters
fimo_thresh = 1e-4

In [5]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [6]:
import torch

In [7]:
pwm_motif = read_meme_pwm_as_numpy(motif_file)
pwm_motif_tensor = torch.from_numpy(pwm_motif.T).float()
motifs_dict = {"motif1": pwm_motif_tensor}

In [8]:
pwm_motif

array([[0.598, 0.116, 0.127, 0.159],
       [0.178, 0.476, 0.09 , 0.256],
       [0.099, 0.632, 0.169, 0.1  ],
       [0.14 , 0.066, 0.697, 0.097],
       [0.583, 0.105, 0.158, 0.154],
       [0.143, 0.137, 0.095, 0.625],
       [0.101, 0.691, 0.069, 0.139],
       [0.108, 0.087, 0.7  , 0.105]])

In [9]:
from tangermeme.tools import fimo

In [10]:
def one_hot_encode(seq):
    """One-hot encode a DNA sequence into shape (len, 4)."""
    mapping = {"A":0, "C":1, "G":2, "T":3}
    arr = np.zeros((len(seq), 4), dtype=np.float32)
    for i, base in enumerate(seq):
        if base in mapping:
            arr[i, mapping[base]] = 1.0
    return arr

In [11]:
# Store motif hit positions for all sequences
positions_edited = []
positions_original = []

for _, row in df.iterrows():
    chrom = row['chrom']
    centered_start = row['centered_start']
    centered_end = row['centered_end']
    
    seq_id = f"{chrom}_{centered_start}_{centered_end}"
    edited_fasta = os.path.join(edited_fasta_dir, f"{seq_id}.fasta")
    original_fasta = os.path.join(original_fasta_dir, f"{seq_id}.fasta")

    if not os.path.exists(edited_fasta) or not os.path.exists(original_fasta):
        print(f"Warning: Missing fasta for {seq_id}, skipping")
        continue
    
    edited_seq = str(next(SeqIO.parse(edited_fasta, "fasta")).seq).upper()
    original_seq = str(next(SeqIO.parse(original_fasta, "fasta")).seq).upper()
    
    # One-hot encode
    ohe_edit = one_hot_encode(edited_seq)
    
    # Trim original to match center length
    start_trim = (len(original_seq) - len(edited_seq)) // 2
    end_trim = start_trim + len(edited_seq)
    orig_seq_trimmed = original_seq[start_trim:end_trim]
    ohe_orig = one_hot_encode(orig_seq_trimmed)  # now also (2168, 4)
    
    # Add batch dimension
    ohe_edit = np.expand_dims(ohe_edit, axis=0)  # (1, 2168, 4)
    ohe_orig = np.expand_dims(ohe_orig, axis=0)  # (1, 2168, 4)
    
    
    edit_hits = fimo.fimo(
            motifs=motifs_dict,
            sequences=ohe_edit,
            threshold=fimo_thresh,
            reverse_complement=False
    )[0]
    
    orig_hits = fimo.fimo(
        motifs=motifs_dict,
        sequences=ohe_orig,
        threshold=fimo_thresh,
        reverse_complement=False
    )[0]
    
    # Store motif positions (midpoint of motif)
    if not edit_hits.empty:
        positions_edited.extend(((edit_hits['start'] + edit_hits['end']) // 2).tolist())

    if not orig_hits.empty:
        positions_original.extend(((orig_hits['start'] + orig_hits['end']) // 2).tolist())

In [12]:
positions_edited

[]

In [ ]:
positions_original